# Transformer 性能分析：动手练习

这份 notebook 是操作讲义，不是运行全部后自动交作业的脚本。保留《作业1.ipynb》不变。
普通 Transformer 基线已提供；计时、PyTorch Profiler、NVTX 和 Nsight 的位置用 `TODO` 标出，请逐项手动加入。

学习顺序：环境 → 普通模型 → Python 计时 → PyTorch Profiler → NVTX → Nsight → 1.1.4 回答。

三种工具的区别：

| 工具 | 主要回答的问题 | 输出 |
|---|---|---|
| Python `default_timer` + CUDA 同步 | 一次操作端到端花多久？ | ms、均值、标准差 |
| `torch.profiler` | 哪些 PyTorch 算子、CUDA kernel 花时间？ | 算子汇总、Chrome trace |
| NVIDIA Nsight Systems | CPU 调度、CUDA API、GPU kernel 如何排列和关联？ | `.nsys-rep` 时间线、kernel 汇总 |

1.1.4 明确要求 Nsight，PyTorch Profiler 是辅助学习，不能代替 nsys 报告。性能分析器本身有开销，不比较“哪种工具跑得快”。

**先保存其他 notebook 并停止 GPU 计算。** 如果残留大量显存，先关闭其他内核。这里的实验不自动启动、停止或清空其他 notebook。

In [1]:
import sys
import shutil
from pathlib import Path
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "不可用")
print("nsys:", shutil.which("nsys"))
print("工作目录:", Path.cwd())
assert torch.cuda.is_available(), "请选择项目 .venv/bin/python 内核"

Python: /home/bygpu/projects/diy-llm/coursework/assignment2-systems/.venv/bin/python
PyTorch: 2.9.1+cu128
CUDA runtime: 12.8
GPU: NVIDIA GeForce RTX 3090
nsys: /usr/local/cuda/bin/nsys
工作目录: /home/bygpu/projects/diy-llm/coursework/assignment2-systems/cs336_systems


## 1. 创建普通 Transformer 基线文件

下格只写入 `training_baseline.py`，不会训练。文件位于本 notebook 工作目录，Linux 终端也能执行。
基线参考作业1：FP32、batch=4、词表10000、small 模型，随机输入；每步只创建新计算图，不重新初始化模型。

先运行原样基线，再手动编辑生成的 `.py` 文件。在 notebook 中可用 VS Code 文件树打开它。
**后续修改了 `.py` 后，不要再次运行下面的 `%%writefile`，否则会覆盖你的修改。**

四种模式：`forward` 是带计算图前向，用于匹配1.1.3；`inference` 是 no_grad 推理；`forward-backward` 不更新参数；`train` 使用本项目自定义 AdamW。
完整训练第一次 warmup 会初始化 AdamW 状态；初始化开销不纳入正式稳态分析。

In [2]:
%%writefile training_baseline.py
import argparse
import gc
import torch
import torch.nn.functional as F
from cs336_basics.model import BasicsTransformerLM
from cs336_basics.optimizer import AdamW

CONFIGS = {
    "small": dict(d_model=768, d_ff=3072, num_layers=12, num_heads=12),
    "medium": dict(d_model=1024, d_ff=4096, num_layers=24, num_heads=16),
    "large": dict(d_model=1280, d_ff=5120, num_layers=36, num_heads=20),
    "xl": dict(d_model=1600, d_ff=6400, num_layers=48, num_heads=25),
    "2.7B": dict(d_model=2560, d_ff=10240, num_layers=32, num_heads=32),
}

def build_state(size="small", context=128, mode="forward"):
    torch.manual_seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model = BasicsTransformerLM(
        vocab_size=10000, context_length=context,
        rope_theta=10000.0, **CONFIGS[size],
    ).to(device="cuda", dtype=torch.float32)
    model.train(mode != "inference")
    x = torch.randint(10000, (4, context), device="cuda")
    y = torch.randint(10000, (4, context), device="cuda")
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.1) if mode == "train" else None
    return dict(model=model, x=x, y=y, optimizer=optimizer)

def run_step(state, mode="forward"):
    model = state["model"]
    model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()

    # TODO A：在前向语句前后加入 default_timer 和 CUDA 同步。
    # TODO C：给前向代码块加 with torch.cuda.nvtx.range("forward")。
    with torch.set_grad_enabled(mode != "inference"):
        logits = model(state["x"])
    torch.cuda.synchronize()

    if mode in ("forward-backward", "train"):
        # TODO C：给 loss 代码块加 NVTX 标记 "loss"。
        loss = F.cross_entropy(logits.reshape(-1, 10000), state["y"].reshape(-1))
        torch.cuda.synchronize()
        # TODO C：给 backward 代码块加 NVTX 标记 "backward"。
        loss.backward()
        torch.cuda.synchronize()
        if mode == "train":
            # TODO C：给 optimizer 代码块加 NVTX 标记 "optimizer"。
            state["optimizer"].step()
            torch.cuda.synchronize()

    # TODO A：返回只含数字的计时字典；不要返回 logits 或 loss。

def release_state(state):
    model = state.get("model")
    if model is not None:
        model.zero_grad(set_to_none=True)
    model = None
    state.clear()
    gc.collect()
    torch.cuda.empty_cache()

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--size", choices=CONFIGS, default="small")
    parser.add_argument("--context", type=int, default=128)
    parser.add_argument("--mode", choices=["forward", "inference", "forward-backward", "train"], default="forward")
    parser.add_argument("--warmups", type=int, default=5)
    parser.add_argument("--steps", type=int, default=10)
    # TODO D：增加 --capture 这个 store_true 参数，再加入采集控制。
    args = parser.parse_args()
    if args.context < 1 or args.warmups < 0 or args.steps < 1:
        parser.error("context >= 1, warmups >= 0, steps >= 1")
    print(vars(args), flush=True)
    state = {}
    try:
        state = build_state(args.size, args.context, args.mode)
        for _ in range(args.warmups):
            run_step(state, args.mode)
        torch.cuda.synchronize()

        # TODO D：采集开始应在这里，在 warmup 之后。
        samples = []
        for _ in range(args.steps):
            # TODO C：给整个单步增加 "step" NVTX 范围。
            samples.append(run_step(state, args.mode))
        torch.cuda.synchronize()
        # TODO D：采集结束应在这里，用 try/finally 保证停止。
        # TODO A：在采集结束后对 samples 内的计时数字求均值和样本标准差。
        print("已完成；当前基线还没有性能统计，请完成 TODO。", flush=True)
    finally:
        release_state(state)

if __name__ == "__main__":
    main()

Writing training_baseline.py


## 2. 先确认普通模型能运行

在项目 `cs336_systems` 目录执行下面单元格。该进程结束会释放自己占用的 GPU 资源。不要先运行最初的反复创建模型的 `run_model()`。
若报 OOM，先检查 `nvidia-smi`；不要把其他进程占满显存导致的失败当作模型配置结论。

In [3]:
import subprocess
subprocess.run([
    sys.executable, "training_baseline.py",
    "--size", "small", "--context", "128",
    "--mode", "forward", "--warmups", "5", "--steps", "10",
], check=True)

{'size': 'small', 'context': 128, 'mode': 'forward', 'warmups': 5, 'steps': 10}
已完成；当前基线还没有性能统计，请完成 TODO。


CompletedProcess(args=['/home/bygpu/projects/diy-llm/coursework/assignment2-systems/.venv/bin/python', 'training_baseline.py', '--size', 'small', '--context', '128', '--mode', 'forward', '--warmups', '5', '--steps', '10'], returncode=0)

## 3. 练习 A：手动加入 Python 标准库计时

编辑 `training_baseline.py`：

1. 顶部增加 `from timeit import default_timer` 和 `import statistics`。
2. 在 `run_step()` 的 TODO A 处，把前向替换成下面结构。不要把创建模型或清理放进计时。

```python
torch.cuda.synchronize()
start = default_timer()
with torch.set_grad_enabled(mode != "inference"):
    logits = model(state["x"])
torch.cuda.synchronize()
forward_ms = (default_timer() - start) * 1000
```

3. 函数最后添加 `return {"forward_ms": forward_ms}`。想额外复习1.1.3，可用相同结构包住 `loss.backward()`，loss计算放在反向计时前。
4. `main()` 的正式循环已把返回值存入 `samples`，在 TODO A 统计处添加：

```python
times = [sample["forward_ms"] for sample in samples]
print("前向均值/ms:", statistics.mean(times))
if len(times) > 1:
    print("前向样本标准差/ms:", statistics.stdev(times))
```

重新运行上一格验证。这是**10个单步耗时**，不分组；预热5次不进入统计。
记录不启用任何 profiler 的前向均值，用于1.1.4(a)。后面增加 NVTX 后，最好也再采一份不启用 nsys 的匹配基准。

| size | context | 模式 | Python均值/ms | Python标准差/ms |
|---|---:|---|---:|---:|
| small | 128 | forward |43.99  | 0.28 |

## 4. 练习 B：在 notebook 中手动使用 PyTorch Profiler

不要把 PyTorch Profiler 与 nsys 同时开启来做工具对比。
先创建一个模型，预热后，只在需要观察的步骤外包裹 `with torch.profiler.profile(...)`。
下格保留空白位置。先运行原样只会完成预热；请按后面的提示替换 TODO。

修改 `.py` 后需要 reload；下面已包含。`release_state()` 在 finally 中释放本格模型。

In [ ]:
import importlib
import training_baseline as baseline
baseline = importlib.reload(baseline)

state = {}
prof = None
try:
    state = baseline.build_state(size="small", context=128, mode="forward")
    for _ in range(5):
        baseline.run_step(state, "forward")
    torch.cuda.synchronize()

    # TODO B：在这里放入下面讲义中的 with profile(...) 代码。
    # 只包住 baseline.run_step(state, "forward")，不要包 build_state 或 warmup。
finally:
    baseline.release_state(state)

将下面片段手动插入上格 TODO B 的位置，并保持缩进：

```python
from torch.profiler import profile, ProfilerActivity
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_stack=True,
) as prof:
    baseline.run_step(state, "forward")
    torch.cuda.synchronize()

print(prof.key_averages().table(
    sort_by="self_cuda_time_total", row_limit=20
))
prof.export_chrome_trace("pytorch_small_ctx128_forward.json")
```

若当前PyTorch提示统计字段名变化，查看表头后使用 `self_device_time_total`。导出的 Chrome trace 可用兼容的 trace viewer 打开。
将 mode 改成 `forward-backward` 或 `train` 时，build_state 和 run_step 的 mode 必须一致。

观察：`aten::...` 是 PyTorch 算子；一个算子可能对应多个 kernel，不能直接拿它与某个 CUDA kernel 当成同一个统计对象。`self` 不含子算子时间；不要把父子算子的累计时间重复相加。
Profiler只采1步是为了先学习界面；与10步报告比较时应换算为每步且匹配范围。

| PyTorch算子 | CPU self时间 | CUDA self时间 | 调用次数 | 初步解释 |
|---|---:|---:|---:|---|
| 待填 | | | | |

## 5. 练习 C：给模型执行阶段加入 NVTX

继续编辑 `training_baseline.py` 的 `run_step()`，手动在对应语句外增加范围，并保留练习A的计时：

```python
with torch.cuda.nvtx.range("forward"):
    # 前向开始计时 → model(x) → 同步 → 前向结束计时
    ...

with torch.cuda.nvtx.range("loss"):
    # 原来的 loss 计算与同步
    ...

with torch.cuda.nvtx.range("backward"):
    # 原来的 loss.backward() 与同步
    ...

with torch.cuda.nvtx.range("optimizer"):
    # 原来的 optimizer.step() 与同步
    ...
```

`...` 是位置示意，必须换成原有代码，不是可直接替换的实现。loss/backward仍放在原来的if分支内；optimizer仍只在train分支中。
`main()` 的正式循环改为：

```python
for _ in range(args.steps):
    with torch.cuda.nvtx.range("step"):
        samples.append(run_step(state, args.mode))
```

NVTX范围通常是CPU侧范围，通过CUDA关联找到其发起的GPU kernel。不要把NVTX范围持续时间直接称为“GPU kernel时间之和”。阶段同步让边界易读，但会影响调度节奏；对比时保持相同同步方式。

## 6. 练习 D：用 nsys 只采集正式步骤

继续手动编辑脚本：

1. 在 `parse_args()` 前加一项：

```python
parser.add_argument("--capture", action="store_true")
```

2. 把 `main()` 中 warmup之后的正式循环改为下面结构（前面的warmup保留）：

```python
samples = []
if args.capture:
    torch.cuda.cudart().cudaProfilerStart()
try:
    for _ in range(args.steps):
        with torch.cuda.nvtx.range("step"):
            samples.append(run_step(state, args.mode))
    torch.cuda.synchronize()
finally:
    if args.capture:
        torch.cuda.cudart().cudaProfilerStop()
# 统计和打印放这里，采集结束以后。
```

这一步完成前，不要执行带 `--capture-range=cudaProfilerApi` 的命令，否则没有开始信号，不会得到预期采集。

### Linux终端按顺序执行

```bash
cd ~/projects/diy-llm/coursework/assignment2-systems/cs336_systems
nsys --version
nvidia-smi
mkdir -p nsys_results

# 先运行同一脚本，不开分析器，记录Python计时。
../.venv/bin/python training_baseline.py --size small --context 128 --mode forward --warmups 5 --steps 10

# 再由nsys启动同一脚本；--capture是你刚添加的脚本参数。
nsys profile --trace=cuda,nvtx --sample=none \
  --capture-range=cudaProfilerApi --capture-range-end=stop \
  -o nsys_results/small_ctx128_forward \
  ../.venv/bin/python training_baseline.py \
  --size small --context 128 --mode forward --warmups 5 --steps 10 --capture

# 输出整份报告的kernel汇总。
nsys stats --report cuda_gpu_kern_sum nsys_results/small_ctx128_forward.nsys-rep
```

重复采集时改输出文件名，例如末尾加 `_r2`，避免覆盖。nsys可能生成SQLite等辅助文件，这是正常的。
不要在同一次采集中套用前一节的 `torch.profiler.profile()`。

### 打开桌面报告

在**向日葵所显示的Ubuntu桌面终端**执行：

```bash
cd ~/projects/diy-llm/coursework/assignment2-systems/cs336_systems
nsys-ui nsys_results/small_ctx128_forward.nsys-rep
```

如果再次缺少 `libxcb-cursor.so.0`，安装 `sudo apt install libxcb-cursor0`。
若有Conda动态库冲突，可以单次尝试 `env -u LD_LIBRARY_PATH -u LD_PRELOAD -u QT_PLUGIN_PATH -u QT_QPA_PLATFORM_PLUGIN_PATH /usr/local/cuda/bin/nsys-ui nsys_results/small_ctx128_forward.nsys-rep`。
普通SSH终端未必有图形DISPLAY，不要随意猜测或设置DISPLAY来绕过。

## 7. 1.1.4(a)：前向总耗时是否一致？

1. 使用 `--mode forward`，这与之前带梯度记录的1.1.3前向保持一致。
2. Nsight中展开进程/线程的NVTX行，找到正式的10个 `step/forward`。
3. 查看各 `forward` 的时间跨度，取平均；不要把模型初始化、warmup、10步总时间当作单次结果。
4. 与不启用nsys时的Python前向均值比较。

Python同步墙钟时间包含CPU调度、等待与GPU工作；CUDA kernel累计时间不包含所有这些成分，两者无需完全相等。采集和NVTX标记也可能增加开销。

| size | context | Python前向均值/ms | Nsight前向范围均值/ms | 前向kernel累计时间/步/ms（可选） | 解释 |
|---|---:|---:|---:|---:|---|
| small | 128 | | | | |

**提交1–2句**：报告数值、说明是否接近及合理差异；不要比较“工具谁更快”。

## 8. 1.1.4(b)：哪个 kernel 最耗时？调用了多少次？

先用forward报告；在GUI选取**一个完整前向范围**，使用当前版本界面中的按时间选择/范围过滤查看 CUDA GPU Kernel Summary（Stats Systems View）。确保统计仅来自选定范围；命令行 `nsys stats` 默认是整份报告，不能直接当单步调用次数。

按 **Total Time** 排序，记录：kernel完整名称、累计GPU时间、Instances。这里问累计贡献，不是哪个kernel单次最慢。

然后再采前向+反向（沿用练习D命令，只改下面两项）：

```text
-o nsys_results/small_ctx128_forward_backward
--mode forward-backward
```

筛选一个step，比较最耗时kernel是否变化；分别筛forward/backward也能解释来源。

| size/context | 范围 | 最耗时kernel完整名称 | 累计GPU时间/ms | 调用次数 |
|---|---|---|---:|---:|
| small/128 | 单次forward | | | |
| small/128 | 单次forward+loss+backward | | | |

**提交1–2句**：名称、累计时间、单次前向调用次数，以及加入反向后是否变化。

## 9. 1.1.4(c)：除矩阵乘法外有哪些热点？

无需新写模型代码。复用forward报告，识别非GEMM的归约、逐元素、索引/拷贝等kernel，检查实际占比。
不要预先断言某个算子一定最慢，也不要仅凭kernel名字做结论；必要时增加更细NVTX标记或通过PyTorch Profiler辅助定位。

| 非矩阵乘法kernel/操作 | GPU累计时间/ms | 占所选范围kernel总时间/% | 来源证据 |
|---|---:|---:|---|
| 待填 | | | |

**提交1–2句**：列出实际测到的主要非矩阵乘法热点。

## 10. 1.1.4(d)：完整训练步骤与推理比较

基线已使用项目的 `cs336_basics.optimizer.AdamW`，不要换为torch内置版本。新增阶段标记即可，不必重写模型。
再次使用练习D命令，分别更改输出名和模式：

```text
-o nsys_results/small_ctx128_inference     --mode inference
-o nsys_results/small_ctx128_train         --mode train
```

这两行是需要替换到完整命令中的参数，不是独立Linux命令。
inference使用no_grad；train包含前向、loss、反向、AdamW更新。

在相同口径下计算：矩阵乘法相关kernel累计GPU时间 / 所选step所有kernel累计GPU时间。不要一边用CPU墙钟、一边用GPU kernel时间作分母。
分别筛forward/backward/optimizer解释变化。完整训练的AdamW状态会占显存，可能出现之前只测反向时没有的OOM。

| 模式 | 矩阵乘法GPU时间/ms | 全部kernel GPU时间/ms | 矩阵乘法占比/% | 其他主要kernel |
|---|---:|---:|---:|---|
| inference | | | | |
| train | | | | |

**提交1–2句**：比较占比和新增热点。不要预设占比必然增大或减小。

## 11. 1.1.4(e)：细分attention的QK、softmax、PV

已核对远程实现：`src/cs336_basics/model.py` 的 `scaled_dot_product_attention(Q,K,V,mask=None)` 使用项目自定义softmax。
它可能产生多个归约/逐元素kernel，不能只查找名字带softmax的一行。

先在下格打印真实函数。为不改作业1或包文件，可以把原函数**复制到 `training_baseline.py`**，改名为 `annotated_attention`，保留原有数学计算和mask语义，再手动标记以下四个范围：

```python
with torch.cuda.nvtx.range("attention_qk"):
    attention_scores = einsum(Q, K, "... query d_k, ... key d_k -> ... query key") / math.sqrt(K.shape[-1])
if mask is not None:
    with torch.cuda.nvtx.range("attention_mask"):
        attention_scores = torch.where(mask, attention_scores, float("-inf"))
with torch.cuda.nvtx.range("attention_softmax"):
    attention_weights = softmax(attention_scores, dim=-1)
with torch.cuda.nvtx.range("attention_pv"):
    output = einsum(attention_weights, V, "... query key, ... key d_v -> ... query d_v")
return output
```

这是函数体的位置示例，需要你自行加上函数声明。顶部还需 `import math`、`from einops import einsum`、`from cs336_basics.nn_utils import softmax`、`import cs336_basics.model as model_module`。
在 `main()` 中创建模型之前添加：

```python
model_module.scaled_dot_product_attention = annotated_attention
```

此替换只在该脚本进程有效，不修改包文件或作业1；必须先检查复制函数与原函数输出一致，再采集。可以用随机Q/K/V及上下三角mask，`torch.testing.assert_close` 比较（保存替换前的原函数用于对照）。
重新采集forward报告，聚合一个forward中**全部层**对应NVTX范围所关联的kernel时间。不要把FFN的GEMM加进来；不要把mask时间算作softmax。

设batch为B、head数H、序列长度T、head维度d：

- QKᵀ约 `2*B*H*T*T*d` FLOPs；PV同样，合计约 `4*B*H*T*T*d`。
- softmax操作量随 `B*H*T*T` 增长；exp/div等操作的FLOPs计数约定需注明，不强行给唯一常数。
- 自定义实现通常仍计算完整分数矩阵再mask，不应直接按三角形把矩阵乘法FLOPs减半。

| size/context | QK时间/ms | PV时间/ms | softmax全部kernel时间/ms | matmul/softmax时间比 | FLOPs比较及解释 |
|---|---:|---:|---:|---:|---|
| small/128 | | | | | |

**提交1–2句**：比较实际耗时与运算量差异，讨论归约、内存访问和kernel启动开销。

In [ ]:
import inspect
import cs336_basics.model as model_module
print(inspect.getsource(model_module.scaled_dot_product_attention))

## 12. 扩展为作业要求的完整配置

先用small/128完成上述操作，理解读数后，再逐个修改脚本参数：

```text
size: small / medium / large / xl / 2.7B
context: 128 / 256 / 512 / 1024
```

共20种模型/长度组合，按题目对前向、反向、优化器步骤分析。显存不足时记录OOM及失败阶段/模式，不用缩小模型后的结果替代原配置。
在同一报告里反复采10步，kernel汇总是10步总和；想报告单步就按step范围筛选或在确认步骤一致后除以10。不能把总时间与单次Python均值比较。

| size | context | forward | forward-backward | inference | train | 报告路径/失败阶段 |
|---|---:|---|---|---|---|---|
| small | 128 | 待做 | 待做 | 待做 | 待做 | |

### 工具对比总结（由你填）

| 工具 | 实际测到的范围 | 前向耗时或GPU时间 | 热点 | 与其他工具一致之处/差异原因 |
|---|---|---:|---|---|
| Python同步计时 | 单步前向墙钟 | | 不提供内部热点 | |
| PyTorch Profiler | 算子和kernel；注意self/total | | | |
| Nsight Systems | 时间线、API、kernel、NVTX | | | |

最终1.1.4(a)–(e)每题按PDF写1–2句，填真实结果，不套用“必然加速/必然变慢”的结论。

参考：[Nsight使用文档](https://docs.nvidia.com/nsight-systems/UserGuide/)、[Nsight统计报告说明](https://docs.nvidia.com/nsight-systems/AnalysisGuide/)、[PyTorch Profiler](https://docs.pytorch.org/docs/stable/profiler.html)。